# Post-only trên Kaggle — hậu xử lý video cho nhận dạng

Baseline: **video gốc → H.264/H.265 thật → postprocessor → analyzer đóng băng (R3D-18)**.

**Trước khi chạy:**
1. **Settings → Accelerator = GPU** (P100/T4).
2. **Settings → Internet = ON** (cần để clone repo và tải trọng số torchvision lần đầu).
3. **Add Input** → gắn dataset video của bạn dạng `pool/<tên_lớp>/<video>.mp4`, tên lớp khớp nhãn Kinetics-400.

Code chạy: nhánh `feat/postonly-training` của `github.com/munnn01/postonly`. Kết quả ghi vào `/kaggle/working/runs` và `/kaggle/working/eval` (tự động thành output của notebook).

In [ ]:
import subprocess, sys, torch

print("python", sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU"
subprocess.run(["ffmpeg", "-version"], check=True, capture_output=True)
encoders = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
for name in ("libx264", "libx265"):
    assert name in encoders, f"Thieu encoder {name}"
print("FFmpeg OK: libx264 + libx265 | GPU:", torch.cuda.get_device_name(0))

In [ ]:
import pathlib
import subprocess
import sys

REPO_DIR = "/kaggle/working/postonly"
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "-b", "feat/postonly-training",
                    "https://github.com/munnn01/postonly.git", REPO_DIR], check=True)
# Kaggle da co torch/torchvision/opencv/scipy; --no-deps tranh xung dot opencv he thong
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                "-e", f"{REPO_DIR}[test]"], check=True)
print("Repo san sang tai", REPO_DIR)

In [ ]:
from pathlib import Path

from postonly.core import FrozenAnalyzer

analyzer = FrozenAnalyzer("r3d_18").to("cuda").eval()
categories = set(analyzer.categories)

def class_hits(root):
    try:
        return sum(1 for d in root.iterdir() if d.is_dir() and d.name in categories)
    except OSError:
        return 0

best, best_hits = None, 0
for base in sorted(Path("/kaggle/input").glob("*")):
    for root in [base, *sorted(base.glob("*"))]:
        hits = class_hits(root)
        if hits > best_hits:
            best, best_hits = root, hits
assert best is not None and best_hits >= 2, "Khong tim duoc pool lop video trong /kaggle/input"
DATA_ROOT = best
print(f"DATA_ROOT = {DATA_ROOT} | {best_hits} lop khop Kinetics-400")
print("Neu sai thu muc hoac 0 lop khop: doi ten thu muc lop dung nhu Kinetics-400.")

In [ ]:
import subprocess
import sys

for codec in ("h264", "h265"):
    done = subprocess.run([sys.executable, "scripts/smoke.py", "--codec", codec],
                          cwd="/kaggle/working/postonly", capture_output=True, text=True)
    print(done.stdout)
    done.check_returncode()
print("Smoke pass cho ca hai codec")

In [ ]:
import subprocess
import sys

# Pilot nho: kiem tra loop train/evaluate chay du voi du lieu that truoc khi train day
subprocess.run([sys.executable, "-m", "postonly.cli", "train",
                "--data-root", str(DATA_ROOT), "--output", "/kaggle/working/runs/pilot",
                "--codec", "h264", "--device", "cuda",
                "--limit-train", "12", "--limit-val", "4",
                "--frames", "8", "--stride", "2", "--size", "128",
                "--channels", "8", "--epochs", "1", "--batch-size", "2",
                "--qps", "30", "40", "--bootstrap", "0"], check=True)
print("Pilot OK")

In [ ]:
import subprocess
import sys

CONFIG = dict(limit_train=2800, limit_val=700, qps=[30, 35, 40, 45], epochs=10,
              batch_size=4, workers=2, seed=42, lr=1e-4, mse_weight=0.1,
              bootstrap=10000)

def train_codec(codec):
    out = f"/kaggle/working/runs/{codec}"
    cmd = [sys.executable, "-m", "postonly.cli", "train",
           "--data-root", str(DATA_ROOT), "--output", out, "--codec", codec,
           "--device", "cuda", "--ffmpeg", "ffmpeg",
           "--limit-train", str(CONFIG["limit_train"]),
           "--limit-val", str(CONFIG["limit_val"]),
           "--qps", *[str(q) for q in CONFIG["qps"]],
           "--epochs", str(CONFIG["epochs"]),
           "--batch-size", str(CONFIG["batch_size"]),
           "--workers", str(CONFIG["workers"]),
           "--seed", str(CONFIG["seed"]), "--lr", str(CONFIG["lr"]),
           "--mse-weight", str(CONFIG["mse_weight"]),
           "--bootstrap", str(CONFIG["bootstrap"])]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

print("Cau hinh:", CONFIG)

In [ ]:
train_codec("h264")

In [ ]:
train_codec("h265")

In [ ]:
import json
from pathlib import Path

summary = {}
for codec in ("h264", "h265"):
    run = Path(f"/kaggle/working/runs/{codec}")
    out = Path(f"/kaggle/working/eval/{codec}_validation")
    subprocess.run([sys.executable, "-m", "postonly.cli", "evaluate",
                    "--data-root", str(DATA_ROOT), "--checkpoint", str(run / "best.pt"),
                    "--manifest", str(run / "split_manifest.json"), "--output", str(out),
                    "--bootstrap", str(CONFIG["bootstrap"]), "--device", "cuda"], check=True)
    summary[codec] = json.loads((out / "bd_rate.json").read_text())

for codec, result in summary.items():
    print(f"[{codec}] BD-rate(top1) = {result['bd_rate_percent']} "
          f"| mien top1 giao nhau: {result['quality_min']} -> {result['quality_max']}")
    print("  anchor  :", result["curves"]["anchor"])
    print("  postonly:", result["curves"]["postonly"])

## Cách đọc kết quả

- **BD-rate âm** = tiết kiệm bitrate tại cùng Top-1; dương = tốn thêm. Tại cùng QP, post-only luôn dùng **đúng bitstream của anchor** — lãi chỉ đến từ đường cong chất lượng.
- Báo **riêng từng codec**, kèm miền top1 giao nhau. Đường cong phẳng ⇒ `null` (không ép về 0).
- **Validation 700 là dùng để chọn checkpoint** (`best.pt`). Muốn tuyên bố kết quả, đánh giá lại trên test độc lập: cell evaluate bỏ `--manifest` và trỏ `--data-root` sang pool test riêng.
- Bootstrap báo kèm số valid/invalid; nếu `invalid` nhiều, nêu rõ trong báo cáo.

## Sau khi chạy xong

Output notebook chứa: `runs/h264` & `runs/h265` (checkpoint, history, split manifest), `eval/*_validation` (CSV per-video, bd_rate.json). Tải về hoặc lưu thành Kaggle dataset để phân tích tiếp.

Không sửa dataset/test sau khi đã xem kết quả; không chọn/bỏ QP theo nhãn test.